# TP1: Pipeline Datalakehouse para API de clima
Data Engineering - CEL UTN

Modulo 1: Extraccion de datos y almacenamiento en Data Lake

Fecha limite de entrega: Domingo, 9 de Noviembre de 2025, 23:59

Alumno: Matias Falconaro

## Documentación de la API

https://openweathermap.org/api/one-call-3

## Definición del Alcance para el dominio de datos

Criterio de Selección Poblacional: 5 ciudades argentinas más pobladas.

[Ciudades más pobladas de Argentina - Wikipedia](https://en.wikipedia.org/wiki/List_of_cities_in_Argentina_by_population)

Cobertura estratégica de los centros urbanos con mayor densidad poblacional para pronósticos climáticos que impacten a la mayor cantidad de habitantes.

## Arquitectura

![Arquitectura del Pipeline](https://drive.google.com/uc?export=view&id=1rxQImMYwympOK95-Y5dKXiYBXTpOo_N6)

## Decisiones de desarrollo

| Categoría | Decisión | Justificación |
|-----------|----------|---------------|
|**Investigacion APIs**|Sub-foros de data engineering|[StackOverflow](https://stackoverflow.com/questions/29913271/weather-api-for-providing-weather-forecast-based-upon-location)<br> [Reddit](https://www.reddit.com/r/dataengineering/comments/14lcyxr/resources_for_weathergeospatial_data/)<br> [Medium](https://medium.com/@ajeet214/9-free-weather-apis-for-ai-data-projects-6bfc66022e46)|
| **API** | OpenWeatherMap | Mencion recurrente en diferentes foros<br> Plan gratuito disponible<br> Documentación detallada<br> Tiempo de actividad confiable<br> Endpoints claros para clima actual y metadatos |
| **Extracción** | **Incremental** (datos temporales)<br>**Full** (datos estáticos) | Clima actual se actualiza cada 10 minutos<br> Append para preservar histórico<br> Metadatos cambian raramente |
| **Particionamiento** | **Por fecha/hora** (temporales)<br>**Sin particionamiento** (estáticos) | Optimiza consultas temporales<br> Organiza grandes volúmenes de datos<br> Dataset estático es pequeño |
| **Verificación de Infraestructura** | boto3 para validación de bucket MinIO | Valida conectividad S3 antes de ejecutar pipeline<br> Detecta errores de configuración tempranamente<br> Evita fallos silenciosos durante escritura Delta Lake<br> Compatibilidad estándar con cualquier S3-compatible storage<br> Permite realizar verificacion de forma programatica sobre el bucket<br>[StackOverflow](https://stackoverflow.com/questions/51104230/how-to-automate-permissions-for-aws-s3-bucket-objects)|
| **Elecciones Técnicas** | Zona horaria UTC<br> Estructura modular | Evita problemas de zonas horarias<br> Código reutilizable y mantenible |
| **Manejo de Errores** | Validación de respuestas API<br> Sistema de logging<br> Manejo datos faltantes<br> Verificación de directorios | Depuración y trazabilidad |
| **Performance** | Extracción paralelizable<br> Particionamiento temporal<br> Procesamiento por lotes<br> Reintentos automáticos | Optimización consultas<br> Manejo eficiente de memoria<br> Resiliencia a fallos de red |
| **Documentación** | Type hints<br> Docstring minimizados| Reduce la cantidad de lineas de código<br> Mejora la legibilidad de las funciones<br> Evita explicaciones sobre parametros y salidas en forma de string dentro del docstring|
| **Versionamiento Delta Lake** | Archivos históricos preservados | MERGE crea nuevos archivos en lugar de modificar existentes<br> Archivos antiguos marcados como "removed" en transaction log<br> Permite time travel y auditoría de cambios<br> No indica duplicación de datos<br> <br> NOTA: **Comportamiento esperado pero requiere validación adicional**<br> No afecta integridad de datos (0 duplicados verificados)<br> No estoy seguro de que esta sea la forma correcta de hacerlo|
| **Diseño de función de guardado** | Función genérica `save_to_delta_lake()`<br> 3 modos: `append`, `overwrite`, `merge` | Evita duplicación de código entre datos temporales y estáticos<br> Código validado (0 duplicados, constraints activos)<br><br>**Dudas pendientes:**<br> ¿Función única vs separación SOLID (funciones específicas)?<br> ¿MERGE temporal + OVERWRITE estático vs solo MERGE unificado?<br> ¿Clave `(city_id, date, hour)` es correcta para clima?<br> ¿Constraints solo en creación o post-MERGE también?<br> ¿Fallback automático a overwrite si tabla no existe en merge?|


## [1] Dependencias

In [1]:
!pip install boto3 deltalake pandas pyarrow requests treelib

In [2]:
import boto3
import configparser
import json
import logging
import os
import pandas as pd
import pyarrow as pa
import requests
import time

from botocore.exceptions import ClientError
from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError
from datetime import datetime, timezone
from pathlib import Path
from treelib import Tree
from typing import Dict, List, Optional, Union

## [2] Configuraciones del pipeline

In [3]:
def load_config(config_file="/content/pipeline.conf"):
    """Carga configuración desde el archivo pipeline.conf"""
    if not Path(config_file).exists():
        raise FileNotFoundError(f"Configuration file '{config_file}' not found")

    config = configparser.ConfigParser()
    config.read(config_file)

    minio = {k: config.get('minio', k) for k in
             ['endpoint_url', 'access_key', 'secret_key', 'region', 'bucket_name']}

    base = f"s3://{minio['bucket_name']}/{config.get('data_lake', 'base_path')}"

    return {
        'api_key': config.get('api', 'api_key'),
        'base_url': config.get('api', 'base_url'),
        'api_timeout': config.getint('api', 'api_timeout'),
        'max_retries': config.getint('api', 'max_retries'),
        'cities': [c.strip() for c in config.get('api', 'cities').split(',')],
        'minio_config': minio,
        'storage_options': {
            "AWS_ENDPOINT_URL": minio["endpoint_url"],
            "AWS_ACCESS_KEY_ID": minio["access_key"],
            "AWS_SECRET_ACCESS_KEY": minio["secret_key"],
            "AWS_REGION": minio["region"],
            "AWS_ALLOW_HTTP": "true"
        },
        'data_lake_base': base,
        'temporal_data_path': f"{base}/{config.get('data_lake', 'temporal_path')}",
        'static_data_path': f"{base}/{config.get('data_lake', 'static_path')}"
    }

## [3] Gestion de logs

In [4]:
def setup_logging() -> logging.Logger:
    """Configura el sistema de logging para el pipeline."""

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S',
        force=True
    )

    logger = logging.getLogger(__name__)
    logger.info("Pipeline de datos climáticos iniciado")

    return logger

## [4] Funciones de API client

In [5]:
def make_api_request(endpoint: str, params: Dict) -> Optional[Dict]:
    """
    Realiza peticiones a la API con manejo de errores y reintentos automáticos.
    """
    for attempt in range(config['max_retries']):
        try:
            response = requests.get(endpoint, params=params, timeout=config['api_timeout'])
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            if attempt == config['max_retries'] - 1:
                logger.error(f"Timeout al conectar con {endpoint} después de {config['max_retries']} intentos")
                return None
            time.sleep(2 ** attempt)
        except requests.exceptions.RequestException as e:
            if attempt == config['max_retries'] - 1:
                logger.error(f"Error en petición a {endpoint}: {e}")
                return None
            time.sleep(2 ** attempt)

    return None

In [6]:
def get_current_weather(city: str) -> Optional[Dict]:
    """
    Extrae datos del clima actual para una ciudad específica.
    """
    endpoint = f"{config['base_url']}/weather"
    params = {
        "q": city,
        "appid": config['api_key'],
        "units": "metric",
        "lang": "es"
    }

    logger.info(f"Extrayendo datos climáticos para {city}")

    try:
        data = make_api_request(endpoint, params)

        if data and data.get("cod") == 200:
            # Agrego timestamp de extracción
            data['extraction_timestamp'] = datetime.now(timezone.utc).isoformat()
            data['extraction_city'] = city
            return data
        elif data:
            logger.warning(f"API retornó código {data.get('cod')} para {city}: {data.get('message', 'Sin mensaje')}")
            return None
        else:
            logger.warning(f"No se recibió respuesta para {city}")
            return None

    except Exception as e:
        logger.error(f"Error para {city}: {e}")
        return None


In [7]:
def get_city_metadata(city_list: List[str]) -> List[Dict]:
    """
    Genera metadatos de la ciudad a partir de la respuesta de la API.
    Sirve como datos estaticos/referencia.
    """
    metadata = []

    for city in city_list:
        endpoint = f"{config['base_url']}/weather"
        params = {
            "q": city,
            "appid": config['api_key']
        }

        logger.info(f"Extrayendo metadatos para {city}")
        data = make_api_request(endpoint, params)

        if data:
            # Metadatos de ciudades
            metadata.append({
                "city_id": data.get("id"),
                "city_name": data.get("name"),
                "country": data.get("sys", {}).get("country"),
                "latitude": data.get("coord", {}).get("lat"),
                "longitude": data.get("coord", {}).get("lon"),
                "timezone_offset": data.get("timezone"),
                "last_updated": datetime.now(timezone.utc).isoformat()
            })

    return metadata

## [5] Funciones de transformacion de datos

In [8]:
def weather_to_dataframe(weather_data: Union[Dict, List[Dict]]) -> pd.DataFrame:
    """
    Convierte respuestas crudas del clima de la API a DataFrame.
    """
    if isinstance(weather_data, dict):
        weather_data = [weather_data]

    records = []

    for data in weather_data:
        if data is None:
            continue

        # Tiempo de observación del clima (cuando se registró el dato)
        observation_dt = datetime.fromtimestamp(
            data.get("dt", 0), tz=timezone.utc
        )

        record = {
            # Identificadores
            "city_id": data.get("id"),
            "city_name": data.get("name"),
            "country": data.get("sys", {}).get("country"),

            # Condiciones climáticas
            "weather_main": data.get("weather", [{}])[0].get("main"),
            "weather_description": data.get("weather", [{}])[0].get("description"),

            # Datos de temperatura
            "temperature": data.get("main", {}).get("temp"),
            "feels_like": data.get("main", {}).get("feels_like"),
            "temp_min": data.get("main", {}).get("temp_min"),
            "temp_max": data.get("main", {}).get("temp_max"),

            # Otras métricas
            "pressure": data.get("main", {}).get("pressure"),
            "humidity": data.get("main", {}).get("humidity"),
            "visibility": data.get("visibility"),
            "wind_speed": data.get("wind", {}).get("speed"),
            "wind_direction": data.get("wind", {}).get("deg"),
            "cloudiness": data.get("clouds", {}).get("all"),

            # Marcas de tiempo
            "observation_time": datetime.fromtimestamp(
                data.get("dt", 0), tz=timezone.utc
            ).isoformat(),
            "extraction_timestamp": data.get("extraction_timestamp"),

            # Columnas de particionamiento
            "date": observation_dt.strftime("%Y-%m-%d"),
            "hour": observation_dt.strftime("%H")
        }

        records.append(record)

    df = pd.DataFrame(records)
    logger.info(f"DataFrame creado con {len(df)} registros")

    return df

In [9]:
def metadata_to_dataframe(metadata: List[Dict]) -> pd.DataFrame:
    """
    Convierte los metadatos de la ciudad a Dataframe.
    """
    df = pd.DataFrame(metadata)
    logger.info(f"DataFrame de metadatos creado con {len(df)} registros")

    return df

## [6] Configuracion del bucket de alamacenamiento

In [10]:
def test_minio_connection():
    """
    Prueba la conexión a MinIO
    """
    logger.info("Probando conexión a MinIO")

    try:
        logger.info(f"Conectando a: {config['minio_config']['endpoint_url']}")

        s3_client = boto3.client(
            's3',
            endpoint_url=config['minio_config']["endpoint_url"],
            aws_access_key_id=config['minio_config']["access_key"],
            aws_secret_access_key=config['minio_config']["secret_key"],
            region_name=config['minio_config']["region"]
        )

        # Listo buckets
        response = s3_client.list_buckets()
        buckets = [b['Name'] for b in response['Buckets']]

        logger.info("Conexión a MinIO exitosa")
        logger.info(f"Número de buckets: {len(buckets)}")
        logger.info(f"Buckets disponibles: {buckets}")
        return True

    except Exception as e:
        logger.error(f"Error conectando a MinIO: {e}")
        logger.error("Verificar: URL, credenciales, y que MinIO no este caido")
        return False

## [7] Funcion de almacenamiento - Data Lake




In [11]:
def save_to_delta_lake(df: pd.DataFrame,
                      base_path: str,
                      partition_cols: Optional[List[str]] = None,
                      mode: str = "append",
                      merge_predicate: Optional[str] = None) -> None:
    """
    Guarda un DataFrame en formato Delta Lake en MinIO/S3.

    Args:
        df: DataFrame a guardar
        base_path: Ruta en S3/MinIO donde se guardará la tabla
        partition_cols: Columnas por las cuales particionar (opcional)
        mode: Modo de escritura - 'append', 'overwrite' o 'merge' (default: 'append')
        merge_predicate: Condición SQL para merge (requerido solo si mode='merge')

    Modos soportados:
        - 'append': Agrega nuevos registros sin verificar duplicados
        - 'overwrite': Reemplaza todos los datos existentes
        - 'merge': Actualiza registros existentes e inserta nuevos (upsert)

    Aplica constraints de integridad automáticamente en la primera creación.
    """
    try:
        logger.info(f"Guardando {len(df)} registros en {base_path} (modo: {mode})")

        # Convierto a PyArrow
        pa_table = pa.Table.from_pandas(df)

        if mode == "merge":
            if not merge_predicate:
                raise ValueError("merge_predicate es requerido para mode='merge'")

            try:
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                dt.merge(
                    source=df,
                    predicate=merge_predicate,
                    source_alias="source",
                    target_alias="target"
                ).when_matched_update_all().when_not_matched_insert_all().execute()
                logger.info(f"MERGE exitoso: {len(df)} registros procesados")

            except TableNotFoundError:
                logger.info("Tabla no existe, creando inicialmente")
                write_deltalake(
                    base_path,
                    pa_table,
                    mode="overwrite",
                    partition_by=partition_cols,
                    storage_options=config['storage_options']
                )
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                _apply_constraints(dt, base_path)
                logger.info(f"Tabla creada: {len(df)} registros")

        elif mode in ["append", "overwrite"]:
            write_deltalake(
                base_path,
                pa_table,
                mode=mode,
                partition_by=partition_cols,
                storage_options=config['storage_options']
            )
            logger.info(f"{mode.upper()} exitoso: {len(df)} registros")

            # Aplico constraints solo en overwrite (primera creación)
            if mode == "overwrite":
                dt = DeltaTable(base_path, storage_options=config['storage_options'])
                _apply_constraints(dt, base_path)

        else:
            raise ValueError(f"Modo no soportado: {mode}. Usar: append, overwrite o merge")

    except Exception as e:
        logger.error(f"Error escribiendo en Delta Lake ({base_path}): {e}")
        raise


def _apply_constraints(dt: DeltaTable, base_path: str) -> None:
    """
    Aplica constraints de integridad de forma individual.
    """
    constraints = {}

    if "temporal" in base_path:
        constraints = {
            "pk_not_null": "city_id IS NOT NULL AND date IS NOT NULL AND hour IS NOT NULL",
            "valid_humidity_range": "humidity >= 0 AND humidity <= 100"
        }

    elif "metadata" in base_path:
        constraints = {
            "city_id_not_null": "city_id IS NOT NULL",
            "valid_coordinates": "latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180"
        }

    # Aplico cada constraint individualmente
    for constraint_name, constraint_expr in constraints.items():
        try:
            dt.alter.add_constraint({constraint_name: constraint_expr})
            logger.info(f"Constraint '{constraint_name}' aplicado exitosamente")
        except Exception as e:
            logger.warning(f"Error aplicando constraint '{constraint_name}': {e}")

## [8] Uso con datos temporales (Clima)

In [12]:
def extract_and_save_weather_data() -> bool:
    """Extrae datos climáticos actuales y los guarda en Delta Lake."""

    weather_data = []
    for city in config['cities']:
        data = get_current_weather(city)
        if data:
            weather_data.append(data)

    if weather_data:
        df_weather = weather_to_dataframe(weather_data)

        # Validacion basica
        if df_weather.empty:
            logger.warning("No hay datos para guardar")
            return False
        elif df_weather[['city_id', 'observation_time']].isnull().any().any():
            logger.error("Datos incompletos detectados - abortando escritura")
            return False
        else:
            save_to_delta_lake(
                df=df_weather,
                base_path=config['temporal_data_path'],
                partition_cols=["date", "hour"],
                mode="merge",
                # Claves de negocio: date, hour y city_id
                merge_predicate="target.city_id = source.city_id AND target.date = source.date AND target.hour = source.hour"
            )
            logger.info(f"Pipeline ejecutado exitosamente: {len(df_weather)} registros procesados")
            return True
    else:
        logger.warning("No se obtuvieron datos climáticos")
        return False

## [9] Uso con datos estaticos (Metadatos)

In [13]:
def refresh_city_metadata() -> bool:
    """Actualiza los metadatos de ciudades en Delta Lake (full refresh)."""

    metadata = get_city_metadata(config['cities'])

    if metadata:
        df_metadata = metadata_to_dataframe(metadata)
        save_to_delta_lake(
            df=df_metadata,
            base_path=config['static_data_path'],
            mode="overwrite"  # Full refresh para datos estáticos
        )
        logger.info(f"Metadatos actualizados exitosamente: {len(df_metadata)} ciudades")
        return True
    else:
        logger.warning("No se obtuvieron metadatos de ciudades")
        return False

## [10] Verificaciones

In [14]:
def verify_delta_table(path, table_name):
    """
    Verificación de tablas Delta
    NOTA 1: Logger para el flujo del proceso, print/display para visualización de datos.
    NOTA 2: Función temporal para 1ra entrega (extracción y Datalakehouse).
    """
    logger.info(f"Verificando: {table_name}")

    try:
        dt = DeltaTable(path, storage_options=config['storage_options'])
        df = dt.to_pandas()

        logger.info(f"CONEXIÓN EXITOSA - Registros: {len(df):,}\n")

        # Info general
        df.info()
        print()

        # Datos
        # Aunque actualmente solo hay 5 registros, se aplica head(5)
        # anticipando el crecimiento del dataset en futuras entregas del pipeline
        print("PRIMERAS 5 FILAS:")
        display(df.head(5))
        print("\n")

        # Estadísticas
        numeric_cols = df.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            print("ESTADÍSTICAS:")
            display(df[numeric_cols].describe())
            print()

        # Valido merge (Duplicados)
        if 'city_id' in df.columns:
            key_cols = ['city_id', 'date', 'hour'] if 'date' in df.columns else ['city_id']
            duplicados = df.duplicated(subset=key_cols).sum()
            print(f"Duplicados: {duplicados}")
            print()

        return True

    except Exception as e:
        logger.error(f"Error verificando {table_name}: {e}")
        return False


def show_bucket_tree():
    """Muestra árbol completo del data lake"""

    print("ESTRUCTURA DEL DATA LAKE")
    print()

    s3 = boto3.client('s3',
                      endpoint_url=config['minio_config']["endpoint_url"],
                      aws_access_key_id=config['minio_config']["access_key"],
                      aws_secret_access_key=config['minio_config']["secret_key"])

    bucket = config['minio_config']["bucket_name"]

    # Creo árbol
    tree = Tree()
    tree.create_node(bucket, bucket)

    # Proceso objetos
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=bucket):
        for obj in page.get('Contents', []):
            parts = obj['Key'].split('/')
            parent = bucket

            for i, part in enumerate(parts):
                node_id = '/'.join(parts[:i+1])

                if not tree.contains(node_id):
                    tree.create_node(part, node_id, parent=parent)

                parent = node_id

    tree.show()

## [11] Orquestacion del pipeline

In [15]:
# PREPARACION DEL ENTORNO DE TRABAJO
config = load_config()  # Cargo configuración
logger = setup_logging()  # Configuro logs
test_minio_connection()  # Pruebo conexión

# PROCESO DE EXTRACCION Y ESTANDARIZACION DE DATOS
extract_and_save_weather_data()  # Extraigo y guardo datos climaticos
refresh_city_metadata()  # Refresco metadatos de ciudades

logger.info("Iniciando verificación\n")

# VERIFICACIONES y REPORTE DEL PROCESO
verify_delta_table(config['temporal_data_path'], "Datos Climáticos")  # Verifico Delta Table para datos climaticos
print("-" * 60 + "\n")
verify_delta_table(config['static_data_path'], "Metadatos Ciudades")  # Verifico Delta Table para metadatos de ciudades
print("-" * 60 + "\n")
show_bucket_tree()  # Muestro estructura del Datalakehouse

logger.info("Verificación completa.")

20:13:05 - INFO - Pipeline de datos climáticos iniciado
20:13:05 - INFO - Probando conexión a MinIO
20:13:05 - INFO - Conectando a: http://31.97.241.212:9000
20:13:05 - INFO - Conexión a MinIO exitosa
20:13:05 - INFO - Número de buckets: 1
20:13:05 - INFO - Buckets disponibles: ['matiasfalconaro-bucket']
20:13:05 - INFO - Extrayendo datos climáticos para Buenos Aires
20:13:05 - INFO - Extrayendo datos climáticos para Córdoba
20:13:06 - INFO - Extrayendo datos climáticos para Rosario
20:13:06 - INFO - Extrayendo datos climáticos para La Plata
20:13:06 - INFO - Extrayendo datos climáticos para Mar del Plata
20:13:06 - INFO - DataFrame creado con 5 registros
20:13:06 - INFO - Guardando 5 registros en s3://matiasfalconaro-bucket/data_lake/weather_temporal (modo: merge)
20:13:06 - INFO - Tabla no existe, creando inicialmente
20:13:12 - INFO - Constraint 'pk_not_null' aplicado exitosamente
20:13:13 - INFO - Constraint 'valid_humidity_range' aplicado exitosamente
20:13:13 - INFO - Tabla cread

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   city_id               5 non-null      int64  
 1   city_name             5 non-null      object 
 2   country               5 non-null      object 
 3   weather_main          5 non-null      object 
 4   weather_description   5 non-null      object 
 5   temperature           5 non-null      float64
 6   feels_like            5 non-null      float64
 7   temp_min              5 non-null      float64
 8   temp_max              5 non-null      float64
 9   pressure              5 non-null      int64  
 10  humidity              5 non-null      int64  
 11  visibility            5 non-null      int64  
 12  wind_speed            5 non-null      float64
 13  wind_direction        5 non-null      int64  
 14  cloudiness            5 non-null      int64  
 15  observation_time      5 non

,city_id,city_name,country,weather_main,weather_description,temperature,feels_like,temp_min,temp_max,pressure,humidity,visibility,wind_speed,wind_direction,cloudiness,observation_time,extraction_timestamp,date,hour
0,3435910,Buenos Aires,AR,Clear,cielo claro,25.97,25.97,25.53,26.63,1018,40,10000,0.45,270,0,2025-11-10T20:09:15+00:00,2025-11-10T20:13:05.977048+00:00,2025-11-10,20
1,3860259,Córdoba,AR,Clouds,nubes,25.13,24.69,24.55,25.13,1013,38,10000,4.32,33,99,2025-11-10T20:08:44+00:00,2025-11-10T20:13:06.029272+00:00,2025-11-10,20
2,3838583,Rosario,AR,Clear,cielo claro,28.14,27.79,26.17,28.86,1013,40,10000,5.66,30,0,2025-11-10T20:13:06+00:00,2025-11-10T20:13:06.087069+00:00,2025-11-10,20
3,3432043,La Plata,AR,Clear,cielo claro,24.96,24.84,24.96,24.96,1018,51,10000,4.55,73,8,2025-11-10T20:13:06+00:00,2025-11-10T20:13:06.133962+00:00,2025-11-10,20
4,3430863,Mar del Plata,AR,Clear,cielo claro,17.06,16.54,16.64,18.30,1019,66,10000,4.02,63,0,2025-11-10T20:13:06+00:00,2025-11-10T20:13:06.180118+00:00,2025-11-10,20




ESTADÍSTICAS:


,city_id,temperature,feels_like,temp_min,temp_max,pressure,humidity,visibility,wind_speed,wind_direction,cloudiness
count,5.000000e+00,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.0,5.000000,5.000000,5.000000
mean,3.599532e+06,24.252000,23.96600,23.570000,24.776000,1016.200000,47.000000,10000.0,3.800000,93.800000,21.400000
std,2.282531e+05,4.215005,4.33222,3.921702,3.943796,2.949576,11.789826,0.0,1.972524,100.243204,43.517812
min,3.430863e+06,17.060000,16.54000,16.640000,18.300000,1013.000000,38.000000,10000.0,0.450000,30.000000,0.000000
25%,3.432043e+06,24.960000,24.69000,24.550000,24.960000,1013.000000,40.000000,10000.0,4.020000,33.000000,0.000000
50%,3.435910e+06,25.130000,24.84000,24.960000,25.130000,1018.000000,40.000000,10000.0,4.320000,63.000000,0.000000
75%,3.838583e+06,25.970000,25.97000,25.530000,26.630000,1018.000000,51.000000,10000.0,4.550000,73.000000,8.000000
max,3.860259e+06,28.140000,27.79000,26.170000,28.860000,1019.000000,66.000000,10000.0,5.660000,270.000000,99.000000


...



ESTADÍSTICAS:


,city_id,latitude,longitude,timezone_offset
count,5.000000e+00,5.00000,5.000000,5.0
mean,3.599532e+06,-34.37946,-59.741920,-10800.0
std,2.282531e+05,2.46591,2.754117,0.0
min,3.430863e+06,-38.00230,-64.181100,-10800.0
25%,3.432043e+06,-34.92150,-60.639300,-10800.0
50%,3.435910e+06,-34.61320,-58.377200,-10800.0
75%,3.838583e+06,-32.94680,-57.954500,-10800.0
max,3.860259e+06,-31.41350,-57.557500,-10800.0



Duplicados: 0

------------------------------------------------------------

ESTRUCTURA DEL DATA LAKE



20:13:24 - INFO - Verificación completa.


matiasfalconaro-bucket
└── data_lake
    ├── city_metadata
    │   ├── _delta_log
    │   │   ├── 00000000000000000000.json
    │   │   ├── 00000000000000000001.json
    │   │   └── 00000000000000000002.json
    │   └── part-00000-60bb3e56-47a2-49ef-94c4-0ae6f817acad-c000.snappy.parquet
    └── weather_temporal
        ├── _delta_log
        │   ├── 00000000000000000000.json
        │   ├── 00000000000000000001.json
        │   └── 00000000000000000002.json
        └── date=2025-11-10
            └── hour=20
                └── part-00000-6bc93787-3121-4809-b7ba-f33a9aaf19f9-c000.snappy.parquet

